# RoadSight AI — Colab training run (both distress models)

Two **deliberately separate** distress models, trained in this order:

| | Model | Source | Labels | Answers |
|---|---|---|---|---|
| A | `distress_yoloseg` | Crack-Seg | polygons | *how big is the defect* |
| B | `india_detection` | **RDD2022 India** | boxes | does it work **in India** |

**Do not skip the smoke test.** It is two epochs, costs about two minutes, and
proves the data paths before a 60-epoch run can waste an hour on a bad path.

## Before you start

1. **Runtime → Change runtime type → T4 GPU.**
2. Put these on your Google Drive **once** (the repo has no data committed):

   ```
   roadsight_data.zip     <- crack-seg.zip + CRKWH100_gt/  (~120 MB)
   rdd2022_india/         <- extracted India folder          (~0.5 GB)
   ```

   Do **not** upload `rdd2022-DatasetNinja.tar`. It is 14 GB and the India subset
   that actually gets used is 0.5 GB. Upload the extracted folder instead.
3. Run the cells in order.


## Step 0 — config

Edit the two names to match what you put on Drive, then leave the rest alone.

In [ ]:
from pathlib import Path

# --- the only two lines you should edit -----------------------------
CRACK_SEG_ZIP = "roadsight_data.zip"   # crack-seg.zip + CRKWH100_gt/
INDIA_DIR     = "rdd2022_india"        # extracted India images + manifest.json
# --------------------------------------------------------------------

MYDRIVE  = Path("/content/drive/MyDrive")
REPO     = Path("/content/Ghost-Rider")
EPOCHS   = 60
SMOKE    = 2       # epochs for the smoke test
IMGSZ_CS = 416     # Crack-Seg is close-up 416x416
IMGSZ_IN = 720     # RDD India frames are 720x720
BATCH_CS = 16
BATCH_IN = 8       # 720px needs a smaller batch on a T4

print(f"epochs={EPOCHS} smoke={SMOKE} imgsz={IMGSZ_CS}/{IMGSZ_IN} "
      f"batch={BATCH_CS}/{BATCH_IN}")

## Step 1 — mount Drive, get the repo, install

Three separate failures live here, and all three have bitten this project before:

1. `ModuleNotFoundError: No module named 'training'` — the notebook runs from
   `/content`, not the repo. The `%cd` at the end fixes it.
2. `unzip: cannot create extraction directory` — `unzip -d` creates only the
   **final** directory, never intermediate parents.
3. A runtime restart wipes `/content`, so the clone is made idempotent.


In [ ]:
import subprocess, sys

from google.colab import drive
drive.mount("/content/drive")

# --- repo: clone, or reuse if this runtime already has it -------------
if not (REPO / "training").is_dir():
    cloned = subprocess.run(
        ["git", "clone", "https://github.com/Sujith-RMD/Ghost-Rider.git", str(REPO)],
        capture_output=True, text=True,
    )
    if cloned.returncode != 0:
        print(cloned.stderr[-1500:])
        raise SystemExit(
            "clone failed. If the repo is private, run instead:\n"
            f"  !git clone https://<TOKEN>@github.com/Sujith-RMD/Ghost-Rider.git {REPO}"
        )
    print("cloned repo")
else:
    print("repo already present")

# opencv-python-headless, NOT opencv-python: the plain build ships a GUI stack
# that conflicts with Colab's own and breaks cv2 imports.
!pip install -q "ultralytics>=8.3,<9" opencv-python-headless

# %cd matters: 'python -m training.x' only resolves from the repo root.
# Literal path, NOT '%cd {REPO}': IPython magics do not interpolate f-string
# variables, so '{REPO}' would be treated as a directory name and fail.
%cd /content/Ghost-Rider
import os
sys.path.insert(0, "/content/Ghost-Rider")
assert os.getcwd() == "/content/Ghost-Rider", f"cwd is {os.getcwd()}, expected the repo root"
print("cwd:", os.getcwd())


## Step 2 — locate the data on Drive

**This is where runs have failed before.** A path that looks right but is not
(nested folder, "Shared with me", different spelling) prints nothing useful and
then dies minutes later. So the data is *located* first, and every candidate is
printed for you to eyeball.


In [ ]:
import tarfile

MYDRIVE.mkdir(parents=True, exist_ok=True)


def locate(name):
    """Find `name` anywhere in MyDrive and print every candidate."""
    hits = sorted(MYDRIVE.rglob(name))
    for h in hits:
        print("  found:", h)
    if not hits:
        raise SystemExit(
            f"'{name}' not found under {MYDRIVE}.\n"
            "Open the files panel on the left and check the exact name, and "
            "whether it sits inside another folder."
        )
    return hits[0]


def extract(archive, dest):
    """Extract archive -> dest, creating parents first (the classic bug)."""
    dest = Path(dest)
    dest.mkdir(parents=True, exist_ok=True)
    archive = str(archive)
    print("extracting", archive, "->", dest)
    if archive.endswith(".zip"):
        subprocess.run(["unzip", "-q", "-o", archive, "-d", str(dest)], check=True)
    elif archive.endswith((".tar", ".tar.gz", ".tgz")):
        with tarfile.open(archive) as tf:
            tf.extractall(dest)
    else:
        raise SystemExit(f"don't know how to extract {archive}")
    return dest


print("=== Crack-Seg bundle ===")
crack_zip = locate(CRACK_SEG_ZIP)
print("=== India subset ===")
india_src = locate(INDIA_DIR)


## Step 3 — Crack-Seg (model A)

Already in YOLO polygon format with its own `crack-seg.yaml`, so there is **no
conversion step** — that removes a whole class of bug. It only needs unzipping.


In [ ]:
extract(crack_zip, REPO / "data" / "external")

# Verify here, not two epochs into a training run.
train_imgs = REPO / "data/external/crack-seg/images/train"
yaml = REPO / "data/external/crack-seg/crack-seg.yaml"
for label in ("train", "val", "test"):
    d = REPO / "data/external/crack-seg/images" / label
    print(f"crack-seg {label:5s}: {len(list(d.glob('*.*'))) if d.is_dir() else 0} images")

assert yaml.is_file(), f"missing {yaml}"
assert len(list(train_imgs.glob("*.*"))) > 0, "no train images found"
print("yaml:", yaml)


## Step 4 — RDD2022 India (model B)

India **only** — 7,706 images, 6,165 train / 1,541 dev, 3,934 negatives. The 14 GB
tar is not needed; the extracted folder is what gets used.

The YOLO label files and `rdd2022_india.yaml` are produced by our own script so
the split stays the frozen one from `prepare_india_distress.py`. Building them by
hand would silently give a different split.


In [ ]:
# 1. images + manifest into the repo
if not (REPO / "data/external/rdd2022_india/manifest.json").is_file():
    (REPO / "data" / "external").mkdir(parents=True, exist_ok=True)
    import shutil
    shutil.copytree(str(india_src), str(REPO / "data/external/rdd2022_india"))
    print("copied India subset into repo")

# 2. build YOLO files from the frozen split (never by hand)
!python -m training.train_india_detection --prepare-only

# 3. Verify. Every image must have a label file, even when it is EMPTY: an empty
#    file is how YOLO encodes "no damage here". Dropping them would silently
#    delete the 51% negatives, which is the entire reason this set matters.
from pathlib import Path as P
for split in ("train", "dev"):
    imgs = list((P("data/external/rdd2022_india_yolo/images") / split).glob("*.jpg"))
    lbls = list((P("data/external/rdd2022_india_yolo/labels") / split).glob("*.txt"))
    empty = sum(1 for l in lbls if l.stat().st_size == 0)
    print(f"{split:5s} images={len(imgs):5d} labels={len(lbls):5d} empty={empty:5d}")
    assert imgs, f"{split}: no images"
    assert len(imgs) == len(lbls), f"{split}: image/label count mismatch"


## Step 5 — CRKWH100 cross-source test (comes from the Step 3 bundle)

In [ ]:
gt   = REPO / "data/external/CRKWH100_gt"
imgs = REPO / "data/external/deepcrack/CRKWH100/images"

if not gt.is_dir() or not imgs.is_dir():
    print("CRKWH100 not present -> skipping. Crack-Seg still trains;")
    print("only the Step 8 cross-source evaluation needs these.")
else:
    n_img = len(list(imgs.glob("*.png")))
    n_gt = len(list(gt.rglob("*.bmp")))
    print(f"CRKWH100 images={n_img} gt={n_gt}")
    assert n_img and n_img == n_gt, "CRKWH100 image/GT count mismatch"
    if not (REPO / "data/manifests/crkhw100_test.json").is_file():
        !python -m training.prepare_crkhw100_test \
            --images data/external/deepcrack/CRKWH100/images \
            --gt data/external/CRKWH100_gt/CRKWH100_gt \
            --out data/manifests/crkhw100_test.json
    print("cross-source manifest ready")


## Step 6 — smoke test

Two epochs. Proves the paths and the loader before spending an hour on the real
run. **If a smoke test fails, stop** — do not raise the epoch count and hope.

Two things here are deliberate:

- `--workers 2`. Colab's dataloader default of 8 intermittently deadlocks with
  `RuntimeError: DataLoader worker ... exited unexpectedly`. Two is safe.
- The `{BRACES}` in `!` lines are expanded by IPython from the config cell, so
  editing `EPOCHS`/`IMGSZ_CS`/`BATCH_CS` there changes the real run automatically.
  (They work in `!` lines but **not** in `%line` magics — that is why `%cd` above
  uses a literal path.)


In [ ]:
print("=== smoke A: Crack-Seg ===")
!python -m training.train_distress_yoloseg --data data/external/crack-seg \
    --epochs {SMOKE} --imgsz {IMGSZ_CS} --batch {BATCH_CS} --workers 2 --device 0 --name smoke_cs

print("=== smoke B: RDD India ===")
!python -m training.train_india_detection \
    --epochs {SMOKE} --imgsz {IMGSZ_IN} --batch {BATCH_IN} --workers 2 --device 0 --name smoke_in


## Step 7 — real training

Run one at a time and read the printed JSON before starting the next.

In [ ]:
!python -m training.train_distress_yoloseg --data data/external/crack-seg \
    --epochs {EPOCHS} --imgsz {IMGSZ_CS} --batch {BATCH_CS} --workers 2 --device 0 --name distress_yoloseg


In [ ]:
!python -m training.train_india_detection \
    --epochs {EPOCHS} --imgsz {IMGSZ_IN} --batch {BATCH_IN} --workers 2 --device 0 --name india_detection


## Step 8 — evaluate

Each report carries its own caveats. They are not decoration.

In [ ]:
!mkdir -p reports/ml

# A: cross-source on CRKWH100 - different sensor, resolution and geography.
#    Honest cross-source test, and explicitly NOT an India test.
!python -m training.evaluate_crosssource \
    --weights runs/roadsight/distress_yoloseg/weights/best.pt \
    --manifest data/manifests/crkhw100_test.json \
    --out reports/ml/crosssource_eval.json


In [ ]:
# B: India, on the frozen dev split. Gives mAP50 and the per-class breakdown.
#    The false-positive rate on the 51% negative images is the number to read:
#    it is the first false-alarm figure this project has been able to measure.
#
#    No split= is passed: the yaml already maps val: -> images/dev, so the
#    default validates the frozen development set. Passing split=dev would be
#    wrong because that is not a key in the yaml.
!yolo detect val model=runs/roadsight/india_detection/weights/best.pt \
    data=data/external/rdd2022_india_yolo/rdd2022_india.yaml imgsz=720


## Step 9 — bring the results home

Two destinations, for two different reasons:

- **Google Drive** — the weights. Large and binary; they do not belong in git
  history.
- **The repo** — the reports and the run config. Small text, and it is the
  evidence.


In [ ]:
import shutil

DRIVE_OUT = MYDRIVE / "roadsight_results"
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

for w in ("distress_yoloseg", "india_detection"):
    src = REPO / "runs" / "roadsight" / w / "weights" / "best.pt"
    d = DRIVE_OUT / w
    d.mkdir(exist_ok=True)
    if src.is_file():
        shutil.copy(src, d / "best.pt")
        print("saved", d / "best.pt", f"({src.stat().st_size/1e6:.1f} MB)")
    else:
        print("missing weights:", src)


In [ ]:
# Commit the EVIDENCE only, never the weights.
!git add reports/ml data/external/rdd2022_india/manifest.json data/manifests
!git -c user.email="ml@roadsight" -c user.name="ML role" \
    commit -m "ml: colab training reports (E8 segmentation + E10 India detection)"

print("\nPush when you are ready. Use a token; do NOT paste it into this notebook:")
print("  !git push https://<TOKEN>@github.com/Sujith-RMD/Ghost-Rider.git HEAD")


## If it fails

| Symptom | Cause and fix |
|---|---|
| `clone failed` | Private repo. Use the `https://<TOKEN>@github.com/...` form the cell prints. |
| `'roadsight_data.zip' not found under /content/drive/MyDrive` | Wrong name, or nested in a folder. Read the candidates the cell printed. |
| `unzip: cannot create extraction directory` | Should not happen here — `extract()` creates parents first. Report it if you see it. |
| `ModuleNotFoundError: No module named 'training'` | The `%cd` was skipped or run out of order. Re-run Step 1. |
| `DataLoader worker ... exited` | Colab dataloader quirk. Halve `--batch`; do not raise epochs. |
| CUDA out of memory at 720 | Set `--batch 4`, or drop `--imgsz` to 640. |
| Loss will not drop at all | Overfit one image (batch 1). If that fails, the **labels** are wrong, not the learning rate. |
| Crack-Seg good, CRKWH100 poor | Expected — that is domain shift, not a bug. Diagnose on development data. |
| India mAP near zero | Check the `assert len(imgs) == len(lbls)` cell passed. An all-empty label set would still "train". |
